# Plan de modèle prédictif — Délinquance Geldium (Tata iQ, partie 2)

Ce notebook conceptualise un modèle de prédiction du risque de défaut de paiement à partir des enseignements de l'EDA (partie 1) :

1. Générer la logique du modèle avec la GenAI
2. Justifier le choix du modèle
3. Planifier l'évaluation de la performance et de l'équité
4. Produire le plan (`Model_Plan_Geldium.docx`)

Le code est illustratif : l'objectif est de montrer le raisonnement et le flux de travail, pas de livrer un modèle de production.

---
## Étape 1 — Générer la logique du modèle avec la GenAI

### Prompts GenAI utilisés
- « Esquisse un pipeline de modélisation prédictive pour prévoir les retards de crédit, de la sélection des variables à l'évaluation du modèle. »
- « Propose deux options de modélisation (une simple et une complexe) pour prédire la délinquance, et recommandes-en une. »
- « Explique comment le modèle transforme les variables d'entrée d'un client en une prédiction finale du risque, de l'ingestion des données à la sortie. »
- « Génère un pseudocode clair pour construire un modèle de risque de crédit avec le revenu, l'utilisation du crédit et les paiements manqués. »

### 1.1 Deux options de modélisation

| | Option simple : **régression logistique** | Option complexe : **gradient boosting** (XGBoost / LightGBM) |
|---|---|---|
| Principe | Combinaison linéaire pondérée des variables, transformée en probabilité par la fonction sigmoïde | Ensemble de centaines d'arbres de décision construits séquentiellement, chacun corrigeant les erreurs du précédent |
| Interprétabilité | Élevée : chaque coefficient se lit comme un effet sur le risque (odds ratio) | Faible en natif, nécessite SHAP pour expliquer les décisions |
| Interactions non linéaires | À créer manuellement | Capturées automatiquement |
| Volume de données requis | Fonctionne avec 500 lignes | Risque élevé de surapprentissage sur 500 lignes |
| Usage dans le secteur | Standard des scorecards de crédit réglementées | Utilisé comme modèle challenger ou en détection de fraude |

**Recommandation : la régression logistique** comme modèle principal, avec le gradient boosting comme *challenger* pour vérifier qu'aucune relation non linéaire importante n'est ignorée.

### 1.2 Flux de travail : de l'ingestion à la prédiction

```
Données client (Excel / base Geldium)
   │
   ▼
1. Ingestion et validation ── contrôle du schéma, des plages (Utilisation 0–1,2 ; Score 300–850), de la cohérence âge/ancienneté
   │
   ▼
2. Nettoyage ── normalisation d'Employment_Status, imputation médiane + indicateurs de valeur manquante (cf. EDA)
   │
   ▼
3. Ingénierie des variables ── Missed_6m, Late_6m, Over_Limit, Recent_Missed (Month_6), New_Account (Tenure = 0)
   │
   ▼
4. Prétraitement ── standardisation des numériques, one-hot des catégorielles ; exclusion de Location et Age (équité)
   │
   ▼
5. Modèle ── régression logistique pondérée (class_weight = "balanced", 16 % de délinquants)
   │
   ▼
6. Sortie ── probabilité de défaut (0–1) → bande de risque par rang (Élevé = top 10 %, Moyen = 10–30 %, Faible = reste) → action
```

### 1.3 Pseudocode

```
CHARGER données clients
VALIDER schéma et plages ; SIGNALER les profils incohérents
NETTOYER : normaliser Employment_Status ; imputer Income (médiane par emploi), Loan_Balance, Credit_Score
CRÉER variables : Missed_6m, Late_6m, Recent_Missed, Over_Limit, New_Account
X ← [historique de paiement, Credit_Utilization, Debt_to_Income_Ratio, Credit_Score, Employment_Status, ...]
y ← Delinquent_Account
DIVISER en entraînement (80 %) / test (20 %), stratifié sur y
PIPELINE ← Standardisation(numériques) + OneHot(catégorielles) + RégressionLogistique(poids équilibrés)
ENTRAÎNER PIPELINE sur entraînement (validation croisée 5 plis pour régler C)
POUR chaque client :
    p ← PIPELINE.predict_proba(client)          # probabilité calibrée (CalibratedClassifierCV)
    bande ← "Élevé" si p dans les 10 % les plus risqués ; "Moyen" si dans les 30 % ; sinon "Faible"
    EXPLIQUER : contributions coefficient × valeur des 3 principales variables
RETOURNER p, bande, explication
```

### 1.4 Esquisse de code (illustrative)

In [1]:
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

TARGET = "Delinquent_Account"
df = pd.read_csv("../Delinquency_prediction_cleaned.csv")  # jeu nettoyé à l'étape 2 de l'EDA

# Ingénierie des variables issues de l'EDA
df["Recent_Missed"] = (df.Month_6 == "Missed").astype(int)
df["New_Account"] = (df.Account_Tenure == 0).astype(int)

NUMERIC = ["Missed_Payments", "Missed_6m", "Late_6m", "Recent_Missed", "Credit_Utilization", "Over_Limit",
           "Debt_to_Income_Ratio", "Credit_Score", "Income", "Loan_Balance", "Account_Tenure", "New_Account",
           "Income_missing", "Loan_Balance_missing"]
CATEGORICAL = ["Employment_Status", "Credit_Card_Type"]
# Exclues volontairement : Location et Age (risque de discrimination), Customer_ID (identifiant)

X, y = df[NUMERIC + CATEGORICAL], df[TARGET]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

model = Pipeline([
    ("prep", ColumnTransformer([
        ("num", StandardScaler(), NUMERIC),
        ("cat", OneHotEncoder(drop="first", handle_unknown="ignore"), CATEGORICAL),
    ])),
    ("clf", LogisticRegression(class_weight="balanced", max_iter=1000)),
])
model.fit(X_train, y_train)
print(f"Entraînement : {len(X_train)} clients, test : {len(X_test)} clients")

Entraînement : 400 clients, test : 100 clients


### 1.5 Des variables d'entrée à la décision
Le modèle attribue une probabilité à chaque client du jeu de test. Les bandes de risque sont définies **par rang** (top 10 % = Élevé, 10–30 % = Moyen) plutôt que par un seuil fixe : la pondération des classes gonfle les probabilités absolues, qui devront être recalibrées (`CalibratedClassifierCV`) avant d'utiliser des seuils fixes. Le classement par rang permet aussi d'adapter le volume d'interventions à la capacité des équipes.

In [2]:
ACTIONS = {"Faible": "Aucune action", "Moyen": "Rappel automatique avant échéance", "Élevé": "Contact par un conseiller"}

scored = X_test.copy()
scored["proba_defaut"] = model.predict_proba(X_test)[:, 1].round(3)
rank = scored.proba_defaut.rank(pct=True, method="first")
scored["bande"] = pd.cut(rank, [0, 0.7, 0.9, 1], labels=["Faible", "Moyen", "Élevé"])
scored["action"] = scored.bande.map(ACTIONS)

print(scored.bande.value_counts().reindex(["Élevé", "Moyen", "Faible"]).to_dict())
cols = ["Missed_Payments", "Credit_Utilization", "Debt_to_Income_Ratio", "Credit_Score", "Employment_Status",
        "proba_defaut", "bande", "action"]
scored.sort_values("proba_defaut", ascending=False)[cols].iloc[[0, 1, 15, 16, 98, 99]]

{'Élevé': 10, 'Moyen': 20, 'Faible': 70}


,Missed_Payments,Credit_Utilization,Debt_to_Income_Ratio,Credit_Score,Employment_Status,proba_defaut,bande,action
52,4,0.349340,0.338566,823.0,Employed,0.814,Élevé,Contact par un conseiller
151,1,0.602569,0.229288,524.0,Unemployed,0.793,Élevé,Contact par un conseiller
432,2,0.700700,0.257964,739.0,Employed,0.610,Moyen,Rappel automatique avant échéance
224,0,0.560467,0.282422,535.0,Unemployed,0.606,Moyen,Rappel automatique avant échéance
105,6,0.626524,0.199682,340.0,Retired,0.208,Faible,Aucune action
92,6,0.396749,0.184346,372.0,Retired,0.193,Faible,Aucune action


**Point d'attention.** Les clients classés « Faible » ont ici 6 paiements manqués et un Credit_Score inférieur à 400, alors que les mieux notés figurent parmi les « Élevé ». Le modèle reproduit les relations inversées constatées dans l'EDA, ce qui confirme que **ce pipeline ne doit pas être déployé avant la fiabilisation des données** par Geldium. La logique reste valable ; ce sont les données d'entrée qui posent problème.

### 1.6 Synthèse de l'étape 1

**Type de modèle choisi.** Une **régression logistique pondérée** estime la probabilité qu'un client devienne délinquant à partir d'une combinaison linéaire de ses caractéristiques financières et de son historique de paiement. Cette probabilité est ensuite convertie en bande de risque (Faible / Moyen / Élevé) qui déclenche une action d'intervention graduée. Un modèle de gradient boosting sert de challenger pour détecter d'éventuelles relations non linéaires.

**Les 5 principales variables d'entrée**

| # | Variable | Contribution attendue à la prédiction |
|---|---|---|
| 1 | **Historique de paiement** (`Missed_Payments`, `Missed_6m`, `Recent_Missed`) | Premier prédicteur du risque de crédit : les retards passés et récents annoncent les défauts futurs. À fiabiliser (incohérence relevée dans l'EDA). |
| 2 | **Credit_Utilization** (+ `Over_Limit`) | Une utilisation élevée ou un dépassement de limite signale une tension de trésorerie. |
| 3 | **Debt_to_Income_Ratio** | Mesure la capacité de remboursement : plus le ratio est élevé, moins le client a de marge. |
| 4 | **Credit_Score** | Synthèse externe de la solvabilité, standard dans tous les modèles de crédit. |
| 5 | **Employment_Status** | La stabilité des revenus influence directement la capacité à payer (19,4 % de délinquance chez les chômeurs dans l'EDA). |

Variables secondaires : `Account_Tenure` / `New_Account` (28,6 % de délinquance pour les comptes de moins d'un an), `Credit_Card_Type`, `Income`, `Loan_Balance`.
Variables exclues : `Location` et `Age`, pour éviter une discrimination géographique ou liée à l'âge. Elles sont réservées aux contrôles d'équité (étape 3).

---
## Étape 2 — Justifier le choix du modèle

### Prompts GenAI utilisés
- « Explique les avantages et inconvénients de la régression logistique par rapport aux arbres de décision pour la prévision du risque financier. »
- « Compare les options de modèles pour prédire la délinquance et explique comment elles équilibrent performance et explicabilité. »
- « Explique comment ces modèles répondent aux besoins opérationnels : vitesse, scalabilité et facilité de surveillance. »

### 2.1 Comparaison des options

| Critère | Régression logistique | Arbre de décision | Gradient boosting |
|---|---|---|---|
| **Précision attendue** | Bonne si les relations sont monotones (cas typique du crédit) | Moyenne, instable sur petit échantillon | Souvent la meilleure sur de grands volumes |
| **Interprétabilité** | Coefficients et odds ratios lisibles, codes motifs par client | Règles « si… alors » très lisibles si l'arbre est peu profond | Boîte noire, nécessite SHAP |
| **Conformité réglementaire** | Standard accepté des régulateurs (scorecards) | Acceptable | Exige une documentation et une validation renforcées |
| **Données financières** | Robuste avec 500 lignes, gère le déséquilibre via la pondération | Surapprentissage sans élagage | Surapprentissage élevé sur 500 lignes |
| **Vitesse / scalabilité** | Entraînement et scoring en millisecondes, formule déployable même en SQL | Rapide | Plus lent à entraîner, réglage d'hyperparamètres coûteux |
| **Surveillance** | Simple : suivi de la dérive de chaque coefficient | Simple | Complexe : dérive diffuse sur des centaines d'arbres |

### 2.2 Illustration : l'explicabilité de la régression logistique
Chaque coefficient se convertit en **odds ratio** : une valeur supérieure à 1 augmente le risque, une valeur inférieure à 1 le réduit (pour une hausse d'un écart-type de la variable).

In [3]:
import numpy as np

feature_names = model.named_steps["prep"].get_feature_names_out()
coefs = pd.DataFrame({
    "variable": [f.split("__", 1)[1] for f in feature_names],
    "coefficient": model.named_steps["clf"].coef_[0],
})
coefs["odds_ratio"] = np.exp(coefs.coefficient)
coefs["effet"] = np.where(coefs.coefficient > 0, "↑ risque", "↓ risque")
coefs.reindex(coefs.coefficient.abs().sort_values(ascending=False).index).round(3).reset_index(drop=True)

,variable,coefficient,odds_ratio,effet
0,Credit_Card_Type_Standard,-0.913,0.401,↓ risque
1,Over_Limit,-0.417,0.659,↓ risque
2,Credit_Card_Type_Platinum,-0.388,0.679,↓ risque
3,Credit_Card_Type_Student,-0.344,0.709,↓ risque
4,Credit_Card_Type_Gold,-0.282,0.755,↓ risque
5,Income,0.224,1.251,↑ risque
6,Loan_Balance_missing,0.168,1.183,↑ risque
7,Employment_Status_Retired,-0.162,0.850,↓ risque
8,Debt_to_Income_Ratio,0.161,1.174,↑ risque
9,Credit_Utilization,0.134,1.144,↑ risque


**Ce que révèle la transparence du modèle.** Plusieurs coefficients ont un signe contraire à la logique métier : `Missed_Payments` et `Missed_6m` *réduisent* le risque estimé, `Credit_Score` l'*augmente*, et `Over_Limit` le réduit. Un modèle boîte noire aurait produit des scores sans laisser voir ces incohérences. La régression logistique les rend visibles immédiatement, ce qui confirme les problèmes de données relevés dans l'EDA. C'est un argument de plus en sa faveur : en contexte financier, un modèle doit pouvoir être contrôlé par des experts métier avant tout déploiement.

### 2.3 Codes motifs pour un client
La réglementation (par exemple l'*Equal Credit Opportunity Act* aux États-Unis ou l'article 22 du RGPD en Europe) impose d'expliquer une décision défavorable. Avec la régression logistique, les motifs se calculent directement (coefficient × valeur standardisée) :

In [4]:
def reason_codes(client, top=3):
    x = model.named_steps["prep"].transform(client)
    x = x.toarray()[0] if hasattr(x, "toarray") else x[0]
    contrib = pd.Series(x * model.named_steps["clf"].coef_[0], index=coefs.variable)
    return contrib.sort_values(ascending=False).head(top).round(3)

top_client = scored.sort_values("proba_defaut", ascending=False).index[0]
print(f"Client {top_client} — probabilité {scored.loc[top_client, 'proba_defaut']} — bande {scored.loc[top_client, 'bande']}")
print("Principaux facteurs qui augmentent le risque :")
reason_codes(X_test.loc[[top_client]])

Client 52 — probabilité 0.814 — bande Élevé
Principaux facteurs qui augmentent le risque :


variable
Loan_Balance_missing    0.649
Income                  0.334
Credit_Score            0.148
dtype: float64

### 2.4 Comparaison avec un arbre de décision peu profond
Un arbre de profondeur 3 est aussi lisible. On vérifie sa stabilité en le réentraînant sur 5 échantillons différents :

In [5]:
from sklearn.tree import DecisionTreeClassifier, export_text

tree = Pipeline([("prep", model.named_steps["prep"]),
                 ("clf", DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=42))])
tree.fit(X_train, y_train)
print(export_text(tree.named_steps["clf"], feature_names=list(coefs.variable), show_weights=False))

# Instabilité : la variable de la première coupure change selon l'échantillon d'entraînement
roots = []
for seed in range(5):
    Xa, _, ya, _ = train_test_split(X, y, test_size=0.2, stratify=y, random_state=seed)
    t = DecisionTreeClassifier(max_depth=3, class_weight="balanced", random_state=0)
    t.fit(model.named_steps["prep"].transform(Xa), ya)
    roots.append(coefs.variable.iloc[t.tree_.feature[0]])
print("Variable de la première coupure selon 5 échantillons :", roots)

|--- Credit_Utilization <= -1.30
|   |--- Credit_Score <= 1.46
|   |   |--- class: 0
|   |--- Credit_Score >  1.46
|   |   |--- Debt_to_Income_Ratio <= 0.45
|   |   |   |--- class: 0
|   |   |--- Debt_to_Income_Ratio >  0.45
|   |   |   |--- class: 1
|--- Credit_Utilization >  -1.30
|   |--- Credit_Utilization <= 1.82
|   |   |--- Credit_Card_Type_Standard <= 0.50
|   |   |   |--- class: 1
|   |   |--- Credit_Card_Type_Standard >  0.50
|   |   |   |--- class: 0
|   |--- Credit_Utilization >  1.82
|   |   |--- class: 0

Variable de la première coupure selon 5 échantillons : ['Credit_Utilization', 'Credit_Utilization', 'Credit_Utilization', 'Credit_Utilization', 'Debt_to_Income_Ratio']


L'arbre est lisible et sa première coupure est assez stable (Credit_Utilization dans 4 cas sur 5), mais il ne produit que quelques segments de clients et donc des scores en paliers. Il présente aussi les mêmes incohérences métier : une utilisation très élevée (> 1,82 écart-type) mène à la classe « non délinquant ». Il reste une bonne alternative pour communiquer des règles simples, mais il est moins adapté qu'une régression logistique pour classer finement les clients par niveau de risque.

### 2.5 Justification

La **régression logistique pondérée** est le modèle le plus adapté aux besoins actuels de Geldium. Dans le crédit à la consommation, chaque décision défavorable doit pouvoir être expliquée au client et au régulateur. Les coefficients de la régression logistique se traduisent directement en odds ratios et en codes motifs (« utilisation élevée du crédit », « paiements manqués récents »), ce qui répond à cette exigence de transparence sans outil supplémentaire. Sa transparence a d'ailleurs permis de repérer immédiatement des coefficients de signe incohérent, confirmant les problèmes de données de l'EDA. Le modèle est aussi adapté aux données : il reste stable sur 500 clients et gère le déséquilibre des classes (16 % de délinquants) par simple pondération, alors qu'un gradient boosting surapprendrait et qu'un arbre de décision, découpant les clients en quelques segments par seuils, perd l'information fine et produit des scores en paliers. Sur le plan opérationnel, il s'entraîne et score en quelques millisecondes, se déploie comme une simple formule (y compris en SQL dans les systèmes existants) et se surveille facilement en suivant la dérive de chaque coefficient. Le compromis est une précision potentiellement inférieure si des relations non linéaires fortes existent ; c'est pourquoi un gradient boosting sera entraîné comme challenger : s'il dépasse nettement la régression logistique en AUC, les interactions qu'il détecte seront ajoutées explicitement au modèle principal.

---
## Étape 3 — Planifier l'évaluation de la performance et de l'équité

### Prompts GenAI utilisés
- « Suggère un ensemble de métriques pour évaluer un modèle de prédiction du risque financier en termes de précision, d'équité et de biais. »
- « Donne des exemples de techniques de réduction des biais en modélisation du risque de crédit. »
- « Comment interpréter ces métriques et décider quand un modèle doit être amélioré avant déploiement ? »

### 3.1 Cadre d'évaluation

| Dimension | Métrique | Interprétation | Seuil d'acceptation |
|---|---|---|---|
| **Discrimination** | AUC-ROC | Probabilité qu'un délinquant soit mieux classé qu'un non-délinquant (0,5 = hasard) | ≥ 0,70 |
| **Détection des défauts** | Rappel (recall) | Part des vrais délinquants repérés par le modèle : manquer un défaut coûte cher | ≥ 0,60 |
| **Qualité des alertes** | Précision | Part des clients signalés qui sont réellement délinquants : évite de solliciter inutilement les bons clients | ≥ 2 × taux de base (0,32) |
| **Équilibre** | Score F1 | Moyenne harmonique précision/rappel, adaptée au déséquilibre des classes | À maximiser |
| **Calibration** | Score de Brier | Les probabilités annoncées correspondent-elles aux taux observés ? | < Brier du modèle naïf (0,134) |
| **Équité** | Ratio d'impact disparate | Taux de signalement du groupe le moins signalé ÷ celui du plus signalé | ≥ 0,80 (règle des 4/5) |
| **Équité** | Écart d'égalité des chances | Différence de rappel entre groupes : chaque groupe doit voir ses délinquants détectés de la même façon | ≤ 0,10 |

L'**accuracy** seule est écartée : un modèle qui prédit « jamais délinquant » atteint 84 % sans aucune utilité.

### 3.2 Protocole
1. Validation croisée stratifiée en 5 plis (500 clients, trop peu pour un simple découpage entraînement/test).
2. Signalement des clients du top 30 % de risque (bandes Moyen + Élevé), comme défini à l'étape 1.
3. Comparaison avec le challenger gradient boosting et avec un modèle naïf.
4. Audit d'équité par tranche d'âge et par ville (variables exclues du modèle mais conservées pour l'audit).

In [6]:
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import brier_score_loss, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict

cv = StratifiedKFold(5, shuffle=True, random_state=42)
challenger = Pipeline([("prep", clone(model.named_steps["prep"])),
                       ("clf", HistGradientBoostingClassifier(max_depth=3, class_weight="balanced", random_state=42))])
candidates = {"Régression logistique": model, "Gradient boosting (challenger)": challenger,
              "Modèle naïf": DummyClassifier(strategy="prior")}

def flag_top(proba, share=0.30):
    return (pd.Series(proba).rank(pct=True, method="first") > 1 - share).astype(int).values

results, oof = [], {}
for name, est in candidates.items():
    proba = cross_val_predict(clone(est), X, y, cv=cv, method="predict_proba")[:, 1]
    oof[name] = proba
    flags = flag_top(proba)
    results.append({"modèle": name, "AUC": roc_auc_score(y, proba), "Rappel": recall_score(y, flags),
                    "Précision": precision_score(y, flags), "F1": f1_score(y, flags), "Brier": brier_score_loss(y, proba)})
pd.DataFrame(results).set_index("modèle").round(3)

,AUC,Rappel,Précision,F1,Brier
modèle,,,,,
Régression logistique,0.492,0.288,0.153,0.200,0.253
Gradient boosting (challenger),0.370,0.162,0.087,0.113,0.228
Modèle naïf,0.500,0.238,0.127,0.165,0.134


### 3.3 Interprétation et règle de décision
On compare chaque métrique à son seuil d'acceptation. Si l'AUC ou le rappel échoue, le modèle **n'est pas déployé** et on retourne à l'étape « données / variables ».

In [7]:
lr = pd.DataFrame(results).set_index("modèle").loc["Régression logistique"]
checks = pd.DataFrame([
    ["AUC", lr.AUC, "≥ 0,70", lr.AUC >= 0.70],
    ["Rappel", lr.Rappel, "≥ 0,60", lr.Rappel >= 0.60],
    ["Précision", lr["Précision"], "≥ 0,32", lr["Précision"] >= 0.32],
    ["Brier", lr.Brier, "< 0,134", lr.Brier < 0.134],
], columns=["métrique", "valeur", "seuil", "OK"]).round(3)
display(checks)
print("Décision :", "DÉPLOYABLE" if checks.OK.all() else "NON DÉPLOYABLE — corriger les données puis réentraîner")

,métrique,valeur,seuil,OK
0,AUC,0.492,"≥ 0,70",False
1,Rappel,0.288,"≥ 0,60",False
2,Précision,0.153,"≥ 0,32",False
3,Brier,0.253,"< 0,134",False


Décision : NON DÉPLOYABLE — corriger les données puis réentraîner


### 3.4 Audit d'équité par groupe

In [8]:
audit = df[["Age", "Location", TARGET]].copy()
audit["flag"] = flag_top(oof["Régression logistique"])
audit["Tranche_age"] = pd.cut(audit.Age, [17, 30, 50, 75], labels=["18–30", "31–50", "51–74"])

def fairness_table(group):
    g = audit.groupby(group, observed=True)
    table = pd.DataFrame({
        "n": g.size(),
        "taux_signalement": g.flag.mean(),
        "rappel": g.apply(lambda s: s.loc[s[TARGET] == 1, "flag"].mean()),
        "taux_faux_positifs": g.apply(lambda s: s.loc[s[TARGET] == 0, "flag"].mean()),
    })
    di = table.taux_signalement.min() / table.taux_signalement.max()
    eo = table.rappel.max() - table.rappel.min()
    print(f"{group} : impact disparate = {di:.2f} ({'OK' if di >= 0.8 else 'ALERTE'}), "
          f"écart de rappel = {eo:.2f} ({'OK' if eo <= 0.10 else 'ALERTE'})")
    return table.round(3)

display(fairness_table("Tranche_age"))
display(fairness_table("Location"))

Tranche_age : impact disparate = 0.84 (OK), écart de rappel = 0.16 (ALERTE)


,n,taux_signalement,rappel,taux_faux_positifs
Tranche_age,,,,
18–30,103,0.311,0.200,0.330
31–50,189,0.270,0.357,0.255
51–74,208,0.322,0.270,0.333


Location : impact disparate = 0.80 (OK), écart de rappel = 0.11 (ALERTE)


,n,taux_signalement,rappel,taux_faux_positifs
Location,,,,
Chicago,103,0.291,0.267,0.295
Houston,95,0.295,0.312,0.291
Los Angeles,107,0.280,0.238,0.291
New York,92,0.283,0.273,0.284
Phoenix,103,0.350,0.353,0.349


### 3.5 Réduction des biais et considérations éthiques

**Techniques de réduction des biais**, de la plus simple à la plus avancée :
- **Avant l'entraînement :** exclure les attributs sensibles (`Age`, `Location`) et vérifier les proxys (par exemple `Credit_Card_Type = Student` corrélé à l'âge) ; rééchantillonner ou repondérer les groupes sous-représentés.
- **Pendant l'entraînement :** ajouter une contrainte d'équité (par exemple `fairlearn.reductions.ExponentiatedGradient` avec parité démographique ou égalité des chances).
- **Après l'entraînement :** ajuster les seuils de signalement par groupe (`ThresholdOptimizer`) si un groupe reste défavorisé.
- **En production :** relancer l'audit chaque mois et suivre la dérive des données (indice PSI > 0,2 ⇒ réentraînement).

**Considérations éthiques**
- Le score sert à **proposer de l'aide** (rappel, plan de paiement), jamais à sanctionner automatiquement : toute décision défavorable reste validée par un conseiller.
- Chaque client signalé dispose de codes motifs explicables et d'un droit de contestation.
- Les données de clients vulnérables (chômeurs, nouveaux comptes) exigent une attention particulière pour éviter que le modèle ne les exclue systématiquement du crédit.
- Les données personnelles sont minimisées et protégées (accès restreint, pas de variable sans lien avec la solvabilité).

### 3.6 Synthèse de la stratégie d'évaluation

La stratégie repose sur une **validation croisée stratifiée en 5 plis** et un tableau de bord de métriques, chacune avec un seuil d'acceptation. L'**AUC** (≥ 0,70) mesure la capacité de classement, le **rappel** (≥ 0,60) la part des défauts détectés, la **précision** la pertinence des alertes, le **F1** l'équilibre des deux et le **score de Brier** la fiabilité des probabilités. L'accuracy est écartée à cause du déséquilibre des classes. L'**équité** est contrôlée par tranche d'âge et par ville grâce au ratio d'impact disparate (≥ 0,80) et à l'écart d'égalité des chances (≤ 0,10).

**Application au jeu actuel :** la régression logistique obtient un AUC de 0,49 et le challenger 0,37, soit pas mieux que le hasard. Le rappel (0,29) et la précision (0,15) sont aussi sous leurs seuils, et le score de Brier (0,25) est moins bon que celui du modèle naïf (0,13). La règle de décision conclut donc **« non déployable »**, ce qui est cohérent avec l'EDA. Côté équité, l'impact disparate passe le seuil (0,84 par âge, 0,80 par ville), mais l'écart de rappel déclenche une alerte (0,16 entre tranches d'âge, 0,11 entre villes). Avec seulement 11 à 37 délinquants par groupe, ces écarts sont peu fiables, mais ils montrent que l'audit doit être relancé sur les données corrigées. Le cadre d'évaluation fonctionne comme prévu : il bloque un modèle qui n'apprend rien d'utile avant qu'il n'atteigne les clients. Une fois les données corrigées par Geldium, le même protocole sera relancé tel quel.